# WaveGrip: Test-set comparison of the DWT CNNs

Tristan Au

This notebook evaluates the three ResNet-152 v1.5 models that read the **same db4 DWT coefficients** with different input layers, on the **held-out test set**. It uses the same metrics as `Evaluation/cnn_evaluation.ipynb`. The training notebooks never touch the test set. All three models share the same 3,071 test trials (20 % of the 15,351 GRABMyo wrist trials, stratified by participant × gesture).

| CNN | Training notebook | Input (C × H × W) | Input layer |
|---|---|---|---|
| **DWT** | `CNN/cnn_dwt.ipynb` | 1 × 12 × 5120 | signed coefficients, bands joined end to end (A5 \| D5 \| D4 \| D3 \| D2); the width axis restarts at every join |
| **DWT time-frequency** | `CNN/cnn_dwt_tf_image.ipynb` | 12 × 5 × 2560 | the same signed coefficients laid out as electrodes (channels) × bands × continuous time (1.95 ms), each band repeated onto the shared grid |
| **DWT envelope** | `CNN/cnn_dwt_envelope.ipynb` | 5 × 12 × 320 | log energy per band on a shared time grid (15.6 ms frames, 78 ms smoothing); bands are channels |

The three models form a ladder. **DWT → time-frequency** changes only the *layout* (continuous, time-aligned axes, same values). **Time-frequency → envelope** additionally turns signed coefficients into smoothed *log energy*.

**What it shows**

1. A training summary for each run (from its `*_info.json` and `*_history.csv`; validation only).
2. A confusion matrix for each CNN. Cells are coloured by row-normalised % (each true gesture's row sums to 100 %) and annotated with the trial count and the %.
3. A table comparing the CNNs on accuracy, plus macro-averaged precision, recall and F1-score (the unweighted mean over the 17 gestures), with each model's change against the DWT CNN.
4. ROC curves, one-vs-rest with macro-averaging: one figure overlaying the three macro-average curves, and one panel per CNN with its 17 per-class curves.

**Inputs are rebuilt exactly as in training**, from `ProcessedDatasets/dwt_wrist/grabmyo_wrist_dwt.h5`, using each run's saved **training-set** normalisation statistics (`*_normalization.npz`) and a float16 round trip (training kept every DWT input on the GPU as float16). Inference runs in float32, so probabilities may differ from training-time outputs in the last few decimals.

**This notebook only reads `CNN/` and `ProcessedDatasets/`.** Outputs go to `Evaluation/outputs/`.

### Running on Google Colab (T4 GPU)

1. Google Drive needs the same layout the training notebooks use under `MyDrive/WaveGrip/`:
   - `ProcessedDatasets/dwt_wrist/grabmyo_wrist_dwt.h5`
   - `CNN/runs/` (or `CNN/Runs/`) containing `dwt_random_resnet152/`, `dwtenv_random_resnet152/` and `dwttf_random_resnet152/`. A run folder may also sit one level deeper, e.g. inside a folder created by a Drive download (`dwttf_random_resnet152-2026…/dwttf_random_resnet152/`); the notebook finds it either way.
2. Open this notebook in Colab: **Runtime → Change runtime type → T4 GPU**.
3. Run all cells. The configuration cell mounts Drive and copies the DWT dataset (3.8 GB) to the Colab machine's local disk once per session. Outputs go to `MyDrive/WaveGrip/Evaluation/outputs/`, so they outlive the session.

The notebook also runs locally from `Evaluation/` with no changes; CPU inference of the DWT model takes several minutes.

### Caching

Test-set probabilities are saved to `outputs/<run>_test_probs.npz` after inference, in the same format as `cnn_evaluation.ipynb` (so the DWT cache is shared with it). Re-running loads them and skips inference. Set `FORCE_INFERENCE = True` to recompute.

## 0. Configuration

In [ ]:
import gc
import json
import shutil
import sys
import time
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import uniform_filter1d
from sklearn.metrics import (accuracy_score, auc, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from torch import nn

FORCE_INFERENCE = False                              # True: ignore cached test probabilities
DATA_FILE = "dwt_wrist/grabmyo_wrist_dwt.h5"         # relative to ProcessedDatasets/ (all three CNNs read it)

# ---- Paths: Google Colab (Drive) or the local project folder --------------------------------------
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/WaveGrip")    # same layout as the local project folder
    src = PROJECT_ROOT / "ProcessedDatasets" / DATA_FILE
    assert src.exists(), f"upload the dataset to {src}"
    H5_PATH = Path("/content/data") / DATA_FILE                # Drive reads are slow: copy once per session
    if not H5_PATH.exists() or H5_PATH.stat().st_size != src.stat().st_size:
        H5_PATH.parent.mkdir(parents=True, exist_ok=True)
        t0 = time.perf_counter()
        shutil.copyfile(src, H5_PATH)
        print(f"copied {src.name} to local disk in {time.perf_counter() - t0:.0f} s")
else:
    PROJECT_ROOT = Path.cwd().parent                           # run from Evaluation/
    H5_PATH = PROJECT_ROOT / "ProcessedDatasets" / DATA_FILE   # read only
assert H5_PATH.exists(), f"dataset not found at {H5_PATH}"
RUN_ROOTS = [PROJECT_ROOT / "CNN" / d for d in ["runs", "Runs"] if (PROJECT_ROOT / "CNN" / d).is_dir()]
assert RUN_ROOTS, f"no CNN/runs (or CNN/Runs) folder under {PROJECT_ROOT}"   # read only
OUT_DIR = PROJECT_ROOT / "Evaluation" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Device ---------------------------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
elif IN_COLAB:
    print("WARNING: no GPU. In Colab use Runtime -> Change runtime type -> T4 GPU")

# ---- The three DWT CNNs (fixed order and fixed colour + line style everywhere) ---------------------
MODELS = {
    "dwt": dict(label="DWT", run="dwt_random_resnet152", input_shape=(1, 12, 5120), batch=16,
                color="#1baf7a", ls="-"),                    # same colour as in cnn_evaluation.ipynb
    "dwttf": dict(label="DWT time-frequency", run="dwttf_random_resnet152", input_shape=(12, 5, 2560), batch=32,
                  color="#e87ba4", ls="-."),
    "dwtenv": dict(label="DWT envelope", run="dwtenv_random_resnet152", input_shape=(5, 12, 320), batch=64,
                   color="#eda100", ls="--"),
}

# ---- Training settings the evaluation must match (from the training notebooks) --------------------
N_CLASSES = 17
SPLIT, TEST_FRACTION, VAL_FRACTION, SEED = "random", 0.2, 0.2, 0
RESNET_LAYERS = (3, 8, 36, 3)
RESNET_WIDTHS = (64, 128, 256, 512)
DWT_BANDS = ["A5", "D5", "D4", "D3", "D2"]           # join order (DWT), row order (time-frequency), channel order (envelope)
N_TIME = 2560                                        # time-frequency grid: 1.95 ms per column
N_FRAMES, SMOOTH_FRAMES, LOG_EPS = 320, 5, 1e-12     # envelope: 15.6 ms frames, 78 ms smoothing
CHUNK = 512                                          # rows per block when streaming the HDF5 file

GESTURE_NAMES = ["Lateral Prehension", "Thumb Adduction", "Thumb and Little Finger Opposition",
                 "Thumb and Index Finger Opposition", "Thumb and Index Finger Extension",
                 "Thumb and Little Finger Extension", "Index and Middle Finger Extension",
                 "Little Finger Extension", "Index Finger Extension", "Thumb Finger Extension",
                 "Wrist Extension", "Wrist Flexion", "Forearm Supination", "Forearm Pronation",
                 "Hand Open", "Hand Close", "Rest"]
CLASS_TICKS = [str(i) for i in range(1, N_CLASSES + 1)]

# ---- Plot style (same as cnn_evaluation.ipynb) ----------------------------------------------------
INK, INK_2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, PER_CLASS_GRAY = "#e5e4e0", "#c3c2b7", "#c3c2b7"
CM_CMAP = LinearSegmentedColormap.from_list(
    "blue_seq", ["#fcfcfb", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
plt.rcParams.update({"font.family": "sans-serif", "axes.edgecolor": AXIS, "axes.labelcolor": INK_2,
                     "xtick.color": INK_2, "ytick.color": INK_2, "axes.titlecolor": INK})


def find_run(run):
    # CNN/runs/<run>/ or CNN/Runs/<run>/, or one level deeper (e.g. a Drive-download folder <run>-2026.../<run>/)
    for base in RUN_ROOTS:
        hits = [base / run] if (base / run / f"{run}_info.json").exists() else sorted(
            p.parent for p in base.glob(f"*/{run}/{run}_info.json"))
        if hits:
            return hits[0]
    raise FileNotFoundError(f"run folder {run}/ not found under {[str(b) for b in RUN_ROOTS]}")


for key, m in MODELS.items():
    m["dir"] = find_run(m["run"])
    print(f"{key:>6}: {m['dir'].relative_to(PROJECT_ROOT)}  "
          f"files: {sorted(p.name.removeprefix(m['run'] + '_') for p in m['dir'].iterdir())}")
gpu = f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""
print(f"data: {H5_PATH}\ndevice {device}{gpu}, torch {torch.__version__}, outputs -> {OUT_DIR}")

## 1. Training summary

From each run's `*_info.json` and `*_history.csv`. These are **training and validation** numbers only (no test data); the best epoch is the one with the lowest validation loss, whose weights are evaluated below.

In [ ]:
rows = {}
for key, m in MODELS.items():
    info = json.loads((m["dir"] / f"{m['run']}_info.json").read_text())
    m["info"] = info
    rows[m["label"]] = {
        "Input (C × H × W)": " × ".join(map(str, info["input_shape"])),
        "Skip connections": info["skip_connections"],
        "Epochs run": info["epochs_run"],
        "Best epoch": info["best_epoch"],
        "Best val loss": info["best_val_loss"],
        "Best val accuracy": info["best_val_acc"],
        "Train accuracy (best epoch)": info["best_train_acc"],
        "Mean train time / epoch (s)": info["mean_epoch_train_s"],
        "Device": info["device"],
    }
training = pd.DataFrame(rows).T
training.index.name = "CNN"
training.style.format({"Best val loss": "{:.4f}", "Best val accuracy": "{:.4f}",
                       "Train accuracy (best epoch)": "{:.4f}", "Mean train time / epoch (s)": "{:.1f}"})

## 2. Test set

The split is rebuilt with the training notebooks' `holdout` and checked against every run's saved `*_split.npz`, so all three CNNs are scored on the same trials.

In [ ]:
with h5py.File(H5_PATH) as f:
    meta = {k: f[k][:].astype(np.int64) for k in ["session", "participant", "gesture", "trial"]}
labels = meta["gesture"] - 1


def holdout(rows, fraction, seed):
    # Copy of the training notebooks' "random" split: stratified by participant x gesture
    strata = meta["participant"][rows] * 100 + meta["gesture"][rows]
    return train_test_split(rows, test_size=fraction, stratify=strata, random_state=seed)


trainval_rows, test_rows = holdout(np.arange(len(labels)), TEST_FRACTION, SEED)
train_rows, val_rows = holdout(trainval_rows, VAL_FRACTION, SEED + 1)
rebuilt = {"train": np.sort(train_rows), "val": np.sort(val_rows), "test": np.sort(test_rows)}
for key, m in MODELS.items():
    saved = np.load(m["dir"] / f"{m['run']}_split.npz")
    assert all(np.array_equal(saved[k], rebuilt[k]) for k in rebuilt), f"{key}: saved split != rebuilt split"

TEST_ROWS = rebuilt["test"]
y_test = labels[TEST_ROWS]
print(f"train {len(rebuilt['train'])}, val {len(rebuilt['val'])}, test {len(TEST_ROWS)} trials; "
      f"test trials per gesture {np.bincount(y_test).min()}-{np.bincount(y_test).max()}; "
      f"split = saved (= rebuilt) for {', '.join(MODELS)}")

## 3. Model: ResNet-152 v1.5

`Bottleneck` and `ResNet152` are copied from the training notebooks so the saved `state_dict` keys line up. The number of input channels (1, 12 or 5) is read from each run's `conv1` weights, and the skip-connection setting from the presence of the `downsample` projections.

In [ ]:
class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_ch, width, stride=1, skip=True):
        super().__init__()
        out_ch = width * self.expansion
        self.conv1 = nn.Conv2d(in_ch, width, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(width)
        self.conv2 = nn.Conv2d(width, width, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(width)
        self.conv3 = nn.Conv2d(width, out_ch, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.skip = skip
        self.downsample = None
        if skip and (stride != 1 or in_ch != out_ch):
            self.downsample = nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                                            nn.BatchNorm2d(out_ch))

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        if self.skip:
            out = out + (x if self.downsample is None else self.downsample(x))
        return self.relu(out)


class ResNet152(nn.Module):
    def __init__(self, in_ch=1, n_classes=N_CLASSES, layers=RESNET_LAYERS, widths=RESNET_WIDTHS, skip=True):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, 64, 7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(3, stride=2, padding=1)
        ch, stages = 64, []
        for i, (n_blocks, width) in enumerate(zip(layers, widths)):
            blocks = []
            for b in range(n_blocks):
                blocks.append(Bottleneck(ch, width, stride=2 if (i > 0 and b == 0) else 1, skip=skip))
                ch = width * Bottleneck.expansion
            stages.append(nn.Sequential(*blocks))
        self.layer1, self.layer2, self.layer3, self.layer4 = stages
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(ch, n_classes)
        self.log_softmax = nn.LogSoftmax(dim=1)

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.log_softmax(self.fc(torch.flatten(self.avgpool(x), 1)))


def load_trained(key):
    # Weights from *_state_dict.pt, or from best_state in *_checkpoint.pt if the state_dict is missing
    m = MODELS[key]
    sd_path, ckpt_path = m["dir"] / f"{m['run']}_state_dict.pt", m["dir"] / f"{m['run']}_checkpoint.pt"
    if sd_path.exists():
        state, source = torch.load(sd_path, map_location="cpu", weights_only=True), sd_path.name
    else:
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True, mmap=True)
        assert ckpt["finished"], f"{key}: checkpoint is from an unfinished run"
        state, source = ckpt["best_state"], f"{ckpt_path.name} [best_state, epoch {ckpt['best_epoch']}]"
        del ckpt
    in_ch = state["conv1.weight"].shape[1]
    assert in_ch == m["input_shape"][0], f"{key}: conv1 has {in_ch} input channels, expected {m['input_shape'][0]}"
    skip = any(".downsample." in k for k in state)
    model = ResNet152(in_ch=in_ch, skip=skip)
    model.load_state_dict(state)
    m["weights_source"], m["skip"] = source, skip
    return model.to(device).eval()

## 4. Test inputs

The test rows' coefficients are read from the DWT file once. Each CNN's inputs are then built from them exactly as in its training notebook, with **that run's** saved training-set statistics:

| CNN | Preprocessing (same as training) |
|---|---|
| DWT | z-score each band per electrode → join A5 \| D5 \| D4 \| D3 \| D2 → float16 → `(B, 1, 12, 5120)` |
| DWT time-frequency | z-score each band per electrode → float16 → repeat each band to 2560 columns (A5/D5 × 8, D4 × 4, D3 × 2, D2 × 1) per batch → `(B, 12, 5, 2560)` |
| DWT envelope | mean squared coefficient per 15.6 ms frame → 5-frame (78 ms) moving average → `log10(· + 1e-12)` → z-score per (band, electrode) → float16 → `(B, 5, 12, 320)` |

Each builder returns a function `get(i)` giving the inputs for test positions `i` as float32.

In [ ]:
def read_test_coefficients():
    is_test = np.zeros(len(labels), bool)
    is_test[TEST_ROWS] = True
    with h5py.File(H5_PATH) as f:
        return {b: np.concatenate([f["dwt"][b][a:a + CHUNK][is_test[a:a + CHUNK]]
                                   for a in range(0, len(labels), CHUNK)]).astype(np.float32) for b in DWT_BANDS}


def zscore_bands(coeffs, norm):
    return [(coeffs[b] - norm[f"{b}_mean"][:, None]) / norm[f"{b}_std"][:, None] for b in DWT_BANDS]


def tf_index(lengths, n_time=N_TIME):
    """Copy of cnn_dwt_tf_image.ipynb: (n_bands, n_time) column of the joined 5120 layout for each image pixel."""
    rows, off = [], 0
    for L in lengths:
        assert n_time % L == 0, (L, n_time)
        rows.append(off + np.arange(n_time) * L // n_time)
        off += L
    return np.stack(rows)


def band_envelope(c, n_frames=N_FRAMES, smooth=SMOOTH_FRAMES, eps=LOG_EPS):
    """Copy of cnn_dwt_envelope.ipynb: DWT coefficients (..., L) -> log10 smoothed energy envelope (..., n_frames)."""
    per_frame = c.shape[-1] // n_frames
    energy = (c.astype(np.float64) ** 2).reshape(*c.shape[:-1], n_frames, per_frame).mean(-1)
    if smooth > 1:
        energy = uniform_filter1d(energy, smooth, axis=-1, mode="nearest")
    return np.log10(energy + eps).astype(np.float32)


def build_inputs(key, coeffs):
    m = MODELS[key]
    norm = np.load(m["dir"] / f"{m['run']}_normalization.npz")
    if key == "dwt":
        x = torch.from_numpy(np.concatenate(zscore_bands(coeffs, norm), axis=-1)).to(torch.float16)   # (n, 12, 5120)
        return lambda i: x[i].float().unsqueeze(1)
    if key == "dwttf":
        x = torch.from_numpy(np.concatenate(zscore_bands(coeffs, norm), axis=-1)).to(torch.float16)   # (n, 12, 5120)
        idx = torch.from_numpy(tf_index([coeffs[b].shape[-1] for b in DWT_BANDS]))
        return lambda i: x[i][:, :, idx].float()                                                      # (B, 12, 5, 2560)
    if key == "dwtenv":
        env = np.stack([(band_envelope(coeffs[b]) - norm[f"{b}_mean"][:, None]) / norm[f"{b}_std"][:, None]
                        for b in DWT_BANDS], axis=1)                                                  # (n, 5, 12, 320)
        x = torch.from_numpy(env.astype(np.float32)).to(torch.float16)
        return lambda i: x[i].float()
    raise KeyError(key)

## 5. Inference on the test set

For each CNN this loads the cached test probabilities if they exist; otherwise it builds the test inputs, loads the weights and runs the model. The output is log-softmax, so the class probabilities are `exp(output)`.

In [ ]:
@torch.inference_mode()
def predict(model, get, n, batch):
    out, t0 = [], time.perf_counter()
    for k, start in enumerate(range(0, n, batch)):
        i = torch.arange(start, min(start + batch, n))
        out.append(model(get(i).to(device)).float().exp().cpu())
        if (k + 1) % 20 == 0 or start + batch >= n:
            print(f"\r  {min(start + batch, n)}/{n} trials, {time.perf_counter() - t0:.0f} s", end="", flush=True)
    print()
    return torch.cat(out).numpy()


probs, coeffs = {}, None
for key, m in MODELS.items():
    cache = OUT_DIR / f"{m['run']}_test_probs.npz"
    if cache.exists() and not FORCE_INFERENCE:
        c = np.load(cache, allow_pickle=False)
        assert np.array_equal(c["test_rows"], TEST_ROWS), f"{cache.name} was made for a different test set"
        probs[key] = c["probs"]
        m["weights_source"], m["skip"] = str(c["weights_source"]), bool(c["skip"])
        print(f"{key}: loaded cached probabilities from {cache.name}")
        continue
    if coeffs is None:
        t0 = time.perf_counter()
        coeffs = read_test_coefficients()
        print(f"read the test coefficients in {time.perf_counter() - t0:.0f} s")
    print(f"{key}: building test inputs ...")
    get = build_inputs(key, coeffs)
    x0 = get(torch.arange(2))
    assert tuple(x0.shape[1:]) == m["input_shape"] and torch.isfinite(x0).all(), x0.shape
    model = load_trained(key)
    print(f"  weights: {m['weights_source']} (skip connections: {m['skip']}), input {m['input_shape']}")
    probs[key] = predict(model, get, len(TEST_ROWS), m["batch"])
    np.savez(cache, probs=probs[key], test_rows=TEST_ROWS, y_true=y_test,
             weights_source=m["weights_source"], skip=m["skip"])
    del model, get
    gc.collect()
del coeffs

for key, p in probs.items():
    assert p.shape == (len(TEST_ROWS), N_CLASSES) and np.allclose(p.sum(1), 1, atol=1e-3), key
preds = {key: p.argmax(1) for key, p in probs.items()}

## 6. Gesture key

The plots label the gestures 1–17. This is the GRABMyo name for each number.

In [ ]:
gesture_key = pd.DataFrame({"Gesture": range(1, N_CLASSES + 1), "Name": GESTURE_NAMES,
                            "Test trials": np.bincount(y_test, minlength=N_CLASSES)}).set_index("Gesture")
gesture_key

## 7. Confusion matrices

One figure per CNN. Rows are the true gesture and columns the predicted gesture. Colour shows the row-normalised % (each row sums to 100 %, so the diagonal is per-gesture recall). Each cell shows the trial count over its %; cells with no trials show a muted `0`.

In [ ]:
def plot_confusion(key):
    m = MODELS[key]
    cm = confusion_matrix(y_test, preds[key], labels=range(N_CLASSES))
    pct = 100 * cm / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(13, 11.5), constrained_layout=True)
    im = ax.imshow(pct, cmap=CM_CMAP, vmin=0, vmax=100)
    for r in range(N_CLASSES):
        for c in range(N_CLASSES):
            if cm[r, c] == 0:
                ax.text(c, r, "0", ha="center", va="center", fontsize=7.5, color=MUTED)
            else:
                ax.text(c, r, f"{cm[r, c]}\n{pct[r, c]:.1f}%", ha="center", va="center", fontsize=7.5,
                        linespacing=1.15, color="#ffffff" if pct[r, c] >= 55 else INK)
    ax.set_xticks(range(N_CLASSES), CLASS_TICKS)
    ax.set_yticks(range(N_CLASSES), CLASS_TICKS)
    ax.set_xlabel("Predicted gesture")
    ax.set_ylabel("True gesture")
    ax.set_xticks(np.arange(-0.5, N_CLASSES), minor=True)
    ax.set_yticks(np.arange(-0.5, N_CLASSES), minor=True)
    ax.grid(which="minor", color="#ffffff", lw=1)
    ax.tick_params(which="minor", length=0)
    ax.spines[:].set_visible(False)
    cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    cbar.set_label("% of true-gesture trials")
    cbar.outline.set_visible(False)
    acc = (preds[key] == y_test).mean()
    ax.set_title(f"{m['label']} ResNet-152 ({' × '.join(map(str, m['input_shape']))} input): confusion matrix "
                 f"on the test set (n = {len(y_test)}, accuracy {acc:.3f})", loc="left", fontsize=13)
    fig.savefig(OUT_DIR / f"{m['run']}_confusion_matrix.png", dpi=150)
    plt.show()

In [ ]:
plot_confusion("dwt")

In [ ]:
plot_confusion("dwttf")

In [ ]:
plot_confusion("dwtenv")

## 8. Metrics table

Accuracy, plus macro-averaged precision, recall and F1-score (the unweighted mean of the 17 per-gesture scores), on the test set. The best value in each column is in bold. Below the table, each model's change against the DWT CNN is printed in percentage points. The table is saved as `outputs/dwt_cnns_test_metrics.csv`.

In [ ]:
metrics = pd.DataFrame({
    MODELS[key]["label"]: {
        "Accuracy": accuracy_score(y_test, preds[key]),
        "Precision (macro)": precision_score(y_test, preds[key], average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_test, preds[key], average="macro", zero_division=0),
        "F1-score (macro)": f1_score(y_test, preds[key], average="macro", zero_division=0),
    } for key in MODELS}).T
metrics.index.name = "CNN"
metrics.to_csv(OUT_DIR / "dwt_cnns_test_metrics.csv", float_format="%.6f")

base = metrics.loc[MODELS["dwt"]["label"]]
for key in ["dwttf", "dwtenv"]:
    d = metrics.loc[MODELS[key]["label"]] - base
    print(f"{MODELS[key]['label']:<19} vs DWT: " + ", ".join(f"{k} {100 * v:+.1f} pp" for k, v in d.items()))
metrics.style.format("{:.4f}").highlight_max(axis=0, props="font-weight: bold")

## 9. ROC curves: one-vs-rest, macro-averaged

For each gesture, the CNN's probability for that gesture is scored as a binary classifier (that gesture vs. the other 16), giving 17 one-vs-rest ROC curves. The **macro-average ROC curve** is the mean of the 17 TPRs, each linearly interpolated onto the union of their FPR points. The **macro AUC** in the legends is `roc_auc_score(..., multi_class="ovr", average="macro")`, the unweighted mean of the 17 per-class AUCs. The area under the averaged curve is printed alongside it; the two agree closely but are not defined identically.

In [ ]:
Y_test = label_binarize(y_test, classes=range(N_CLASSES))
roc = {}
for key, p in probs.items():
    per_class = [roc_curve(Y_test[:, c], p[:, c])[:2] for c in range(N_CLASSES)]
    fpr_grid = np.unique(np.concatenate([fpr for fpr, _ in per_class]))
    mean_tpr = np.mean([np.interp(fpr_grid, fpr, tpr) for fpr, tpr in per_class], axis=0)
    roc[key] = dict(per_class=per_class, fpr=fpr_grid, tpr=mean_tpr,
                    auc=roc_auc_score(y_test, p, multi_class="ovr", average="macro"),
                    curve_auc=auc(fpr_grid, mean_tpr))
    print(f"{MODELS[key]['label']:<19} macro AUC (mean of per-class AUCs) {roc[key]['auc']:.4f} | "
          f"area under macro-average curve {roc[key]['curve_auc']:.4f}")


def style_roc_axes(ax):
    ax.plot([0, 1], [0, 1], color=MUTED, ls=":", lw=1, label="Chance (AUC = 0.5)")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.01)
    ax.set_aspect("equal")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.grid(color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)

### 9a. The DWT CNNs compared

Each model has its own colour and line style (DWT solid, time-frequency dash-dot, envelope dashed), so the curves stay distinguishable without relying on colour alone. The inset zooms on the top-left corner, where the curves differ most.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6.5), constrained_layout=True)
inset = ax.inset_axes([0.42, 0.30, 0.52, 0.42])
for key, r in roc.items():
    style = dict(color=MODELS[key]["color"], ls=MODELS[key]["ls"], lw=2)
    ax.plot(r["fpr"], r["tpr"], label=f"{MODELS[key]['label']} (macro AUC = {r['auc']:.4f})", **style)
    inset.plot(r["fpr"], r["tpr"], **style)
style_roc_axes(ax)
inset.set(xlim=(0, 0.2), ylim=(0.6, 1.0))
inset.tick_params(labelsize=8)
inset.grid(color=GRID, lw=0.8)
ax.indicate_inset_zoom(inset, edgecolor=MUTED)
ax.legend(frameon=False, loc="lower right")
ax.set_title("DWT CNNs: macro-averaged one-vs-rest ROC on the test set", loc="left")
fig.savefig(OUT_DIR / "dwt_cnns_roc_macro_ovr_comparison.png", dpi=150)
plt.show()

### 9b. Per-class curves for each CNN

In [ ]:
fig, axes = plt.subplots(1, len(roc), figsize=(6.5 * len(roc), 6.5), constrained_layout=True, squeeze=False)
for ax, (key, r) in zip(axes[0], roc.items()):
    for c, (fpr, tpr) in enumerate(r["per_class"]):
        ax.plot(fpr, tpr, color=PER_CLASS_GRAY, lw=1, label="Per-class OvR (gestures 1-17)" if c == 0 else None)
    ax.plot(r["fpr"], r["tpr"], color=MODELS[key]["color"], ls=MODELS[key]["ls"], lw=2.5,
            label=f"Macro-average (AUC = {r['auc']:.4f})")
    style_roc_axes(ax)
    ax.legend(frameon=False, loc="lower right")
    ax.set_title(f"{MODELS[key]['label']} ({' × '.join(map(str, MODELS[key]['input_shape']))})", loc="left")
fig.suptitle("DWT CNNs: one-vs-rest ROC per gesture and macro-average, test set", x=0.01, ha="left", fontsize=13)
fig.savefig(OUT_DIR / "dwt_cnns_roc_ovr_per_class.png", dpi=150)
plt.show()

## 10. Saved outputs

In [ ]:
for key, m in MODELS.items():
    print(f"{m['label']:<19} weights: {m['weights_source']} (skip connections: {m['skip']})")
names = [f"{m['run']}_{s}" for m in MODELS.values() for s in ["test_probs.npz", "confusion_matrix.png"]]
names += ["dwt_cnns_test_metrics.csv", "dwt_cnns_roc_macro_ovr_comparison.png", "dwt_cnns_roc_ovr_per_class.png"]
print(f"\nin {OUT_DIR}:")
for name in names:
    p = OUT_DIR / name
    print(f"  {name:<45} {p.stat().st_size / 1e3:9.1f} kB" if p.exists() else f"  {name:<45} (missing)")